# Ouroboros Phase 9 (Kaggle variant): planner VLM on free T4

Backup host for when a Colab session drops. Same model, same API, same hand-off.

- Kaggle setup: Settings > Accelerator > **GPU T4 x2**, Settings > **Internet on** (needs a phone-verified Kaggle account). Free quota is about 30 GPU hours a week.
- Kaggle sessions run up to 12 hours; an idle interactive session stops sooner, so keep the last cell running.
- The 7B AWQ model fits on one T4, so the second GPU is unused by default. Set `TP = 2` in the serve cell to split it across both (untested here).
- The model only ever receives placeholder text (and, on `need_visual`, an already-masked JPEG).
- After it prints the new URL and key, update the vault entry "Ouroboros VLM endpoint (VLM_BASE_URL + VLM_API_KEY)". Never paste them in chat or git.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
%%capture pipout
!pip install -q vllm
# vllm pulls a torch built for a newer CUDA than Colab's preinstalled torchaudio; vllm does not need torchaudio.
!pip uninstall -y -q torchaudio


In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-c', 'import vllm, torch; print(vllm.__version__, torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout)

In [ ]:
import os, secrets, subprocess, time, urllib.request, json

MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ'   # fallback: 'Qwen/Qwen2.5-VL-3B-Instruct'
API_KEY = secrets.token_urlsafe(24)          # the tunnel is public, so the endpoint needs a key
MAX_LEN = 8192
TP = 1                                       # 2 = split across both Kaggle T4s

cmd = ['vllm', 'serve', MODEL, '--dtype', 'half', '--max-model-len', str(MAX_LEN),
       '--gpu-memory-utilization', '0.90', '--limit-mm-per-prompt', '{"image": 1}',
       '--api-key', API_KEY, '--port', '8000', '--tensor-parallel-size', str(TP)]
if 'AWQ' in MODEL:
    cmd += ['--quantization', 'awq']
LOG = '/kaggle/working/vllm.log' if os.path.isdir('/kaggle/working') else ('/content/vllm.log' if os.path.isdir('/content') else 'vllm.log')
log = open(LOG, 'w')
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('vLLM exited - see the log cell below')
        time.sleep(5)
print(f'vLLM ready in {time.time() - t0:.0f} s')

In [ ]:
# If the cell above failed, read why.
!tail -n 40 {LOG}


In [ ]:
# Free public HTTPS URL for port 8000 (Cloudflare quick tunnel, no account needed).
import re, subprocess, time, os
if not os.path.exists('cloudflared'):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    line = tun.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('Tunnel:', url)

In [ ]:
# Smoke test through the tunnel with a sanitized-style prompt.
import json, time, urllib.request, socket
# A new quick-tunnel hostname can take a few seconds to resolve.
for _ in range(30):
    try:
        socket.gethostbyname(url.split("//")[1]); break
    except OSError:
        time.sleep(2)
body = {'model': MODEL, 'temperature': 0, 'max_tokens': 120, 'messages': [
  {'role': 'system', 'content': 'Reply with one JSON object: {"op": ..., "element_id": ..., "text": ..., "reason": ...}'},
  {'role': 'user', 'content': 'Task: Fill PAN <PAN_1>\nLegend: <PAN_1>=PAN\nElements:\ne1 textbox "PAN"\ne2 button "Submit"\nNext action JSON:'}]}
req = urllib.request.Request(url + '/v1/chat/completions', data=json.dumps(body).encode(),
                             headers={'content-type': 'application/json', 'authorization': f'Bearer {API_KEY}'})
t = time.time(); out = json.loads(urllib.request.urlopen(req, timeout=120).read())
print(f'{(time.time() - t) * 1000:.0f} ms', out['choices'][0]['message']['content'], out.get('usage'))

## Hand-off

Run the cell below. Put the three values into the Ouroboros server environment (never in chat or git; the key protects a public URL):

```
PLANNER=vlm VLM_BASE_URL=<url>/v1 VLM_MODEL=<model> VLM_API_KEY=<key> uvicorn app.main:app --port 8000
```

Then from the repo: `scripts/phase9-run.sh` runs the end-to-end replay and writes `eval/results/phase9-*.md`.

In [ ]:
print(f'VLM_BASE_URL={url}/v1')
print(f'VLM_MODEL={MODEL}')
print(f'VLM_API_KEY={API_KEY}')

In [ ]:
# Keep the session alive while experiments run (free tiers stop idle sessions). Interrupt to stop.
import time
while server.poll() is None:
    time.sleep(60)